In [1]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

client = bigquery.Client('scaler-sql-476312')

# **SUBQUERIES**

In [2]:
# Question - Analyze purchases made at the market on days when it rained
query = """
SELECT *
FROM `farmers_market.customer_purchases`
WHERE
  market_date IN (
    SELECT market_date
    FROM `farmers_market.market_date_info`
    WHERE market_rain_flag = 1
  )
"""

df = client.query(query).to_dataframe()
df

,product_id,vendor_id,market_date,customer_id,quantity,cost_to_customer_per_qty,transaction_time
0,1,7,2019-07-31,3,2.64,6.99,18:36:00
1,1,7,2019-07-31,8,3.83,6.99,18:34:00
2,1,7,2019-07-31,19,3.69,6.99,18:11:00
3,1,7,2019-07-31,22,1.07,6.99,18:04:00
4,1,7,2019-09-21,6,1.99,6.99,12:17:00
...,...,...,...,...,...,...,...
75,3,7,2020-09-30,1,3.00,0.50,18:18:00
76,3,7,2020-09-30,6,1.00,0.50,18:01:00
77,3,7,2020-09-30,11,2.00,0.50,17:50:00
78,3,7,2020-09-30,14,2.00,0.50,17:05:00


In [3]:
# Question -- List down all the product details where product_category_name contains “Fresh” in it
query = """
SELECT *
FROM `farmers_market.product`
WHERE
  product_category_id IN (
    SELECT product_category_id
    FROM `farmers_market.product_category`
    WHERE product_category_name LIKE "%Fresh%"
  )
"""

df = client.query(query).to_dataframe()
df

,product_id,product_name,product_size,product_category_id,product_qty_type
0,15,Red Potatoes - Small,,1,Null
1,1,Habanero Peppers - Organic,medium,1,lbs
2,2,Jalapeno Peppers - Organic,small,1,lbs
3,9,Sweet Potatoes,medium,1,lbs
4,13,Baby Salad Lettuce Mix,1 lb,1,lbs
5,17,Carrots,sold by weight,1,lbs
6,22,Roma Tomatoes,medium,1,lbs
7,14,Red Potatoes,None,1,null
8,3,Poblano Peppers - Organic,large,1,unit
9,12,Baby Salad Lettuce Mix - Bag,1/2 lb,1,unit


In [4]:
# Question -- List down all the product details where product_category_name contains “Fresh” in it
query = """
SELECT *
FROM `farmers_market.customer`
WHERE
  customer_id NOT IN (
    SELECT DISTINCT customer_id FROM `farmers_market.customer_purchases`
  )
"""

df = client.query(query).to_dataframe()
df

,customer_id,customer_first_name,customer_last_name,customer_zip
0,56,Rohit,Menon,110006
1,55,James,Oliver,122067


In [5]:
# Question -- List down all the product details where product_category_name contains “Fresh” in it
query = """
SELECT *
FROM `farmers_market.product`
WHERE
  product_id NOT IN (
    SELECT product_id FROM `farmers_market.customer_purchases`
  )
"""

df = client.query(query).to_dataframe()
df

,product_id,product_name,product_size,product_category_id,product_qty_type
0,15,Red Potatoes - Small,,1,Null
1,9,Sweet Potatoes,medium,1,lbs
2,13,Baby Salad Lettuce Mix,1 lb,1,lbs
3,17,Carrots,sold by weight,1,lbs
4,22,Roma Tomatoes,medium,1,lbs
5,14,Red Potatoes,None,1,null
6,12,Baby Salad Lettuce Mix - Bag,1/2 lb,1,unit
7,16,Sweet Corn,Ear,1,unit
8,18,Carrots - Organic,bunch,1,unit
9,21,Organic Cherry Tomatoes,pint,1,unit


# CASE WHEN Statement

In [6]:
# Question -- Find out which vendors primarily sell fresh products and which don’t.
query = """
SELECT
  vendor_id,
  vendor_name,
  vendor_type,
  CASE
    WHEN lower(vendor_type) LIKE "%fresh%" THEN "Fresh Seller"
    ELSE "Not a Fresh Seller"
    END AS Fresh_Seller
FROM `farmers_market.vendor`
"""

df = client.query(query).to_dataframe()
df

,vendor_id,vendor_name,vendor_type,Fresh_Seller
0,5,Seashell Clay Shop,Arts & Jewelry,Not a Fresh Seller
1,1,Chris's Sustainable Eggs & Meats,Eggs & Meats,Not a Fresh Seller
2,4,Fields of Corn,Fresh Focused,Fresh Seller
3,7,Marco's Peppers,Fresh Focused,Fresh Seller
4,2,Hernandez Salsa & Veggies,Fresh Variety: Veggies & More,Fresh Seller
5,3,Mountain View Vegetables,Fresh Variety: Veggies & More,Fresh Seller
6,6,Mother's Garlic & Greens,Fresh Variety: Veggies & More,Fresh Seller
7,8,Annie's Pies,Prepared Foods,Not a Fresh Seller
8,9,Mediterranean Bakery,Prepared Foods,Not a Fresh Seller


**IF Statement**

Probably we won't use if statement and its not a best practice because

we don't have nested if in SQL

We have to use one if condition

we need to mention what to do if the condition is false

In [7]:
# Question -- Find out which vendors primarily sell fresh products and which don’t by using if condition
query = """
SELECT
  vendor_id,
  vendor_name,
  vendor_type,
  if(lower(vendor_type) like "%fresh%","Fresh Seller","Not a Fresh Seller") as Fresh_Seller
FROM `farmers_market.vendor`
"""

df = client.query(query).to_dataframe()
df

,vendor_id,vendor_name,vendor_type,Fresh_Seller
0,5,Seashell Clay Shop,Arts & Jewelry,Not a Fresh Seller
1,1,Chris's Sustainable Eggs & Meats,Eggs & Meats,Not a Fresh Seller
2,4,Fields of Corn,Fresh Focused,Fresh Seller
3,7,Marco's Peppers,Fresh Focused,Fresh Seller
4,2,Hernandez Salsa & Veggies,Fresh Variety: Veggies & More,Fresh Seller
5,3,Mountain View Vegetables,Fresh Variety: Veggies & More,Fresh Seller
6,6,Mother's Garlic & Greens,Fresh Variety: Veggies & More,Fresh Seller
7,8,Annie's Pies,Prepared Foods,Not a Fresh Seller
8,9,Mediterranean Bakery,Prepared Foods,Not a Fresh Seller


In [8]:
# Question -- Put the total cost to customer purchases into bins of -
# under $5.00,
# 5.00–9.99
# 10.00–19.99, or
# $20.00 and over.

query = """
SELECT
  customer_id,
  product_id,
  quantity,
  cost_to_customer_per_qty,
  round(quantity * cost_to_customer_per_qty, 2) AS total,
  CASE
    WHEN quantity * cost_to_customer_per_qty < 5 THEN "under $5.00"
    WHEN
      quantity * cost_to_customer_per_qty >= 5
      AND quantity * cost_to_customer_per_qty <= 9.99
      THEN "$5.00 - $9.00"
    WHEN quantity * cost_to_customer_per_qty BETWEEN 10.00 AND 19.00
      THEN "$10.00 - $19.00"
    WHEN quantity * cost_to_customer_per_qty >= 20.00 THEN "$20.00 and over"
    END AS Price_Bins
FROM `farmers_market.customer_purchases`
"""

df = client.query(query).to_dataframe()
df

,customer_id,product_id,quantity,cost_to_customer_per_qty,total,Price_Bins
0,14,1,0.99,6.99,6.92,$5.00 - $9.00
1,14,1,2.18,6.99,15.24,$10.00 - $19.00
2,15,1,1.53,6.99,10.69,$10.00 - $19.00
3,16,1,2.02,6.99,14.12,$10.00 - $19.00
4,22,1,0.66,6.99,4.61,under $5.00
...,...,...,...,...,...,...
998,8,4,1.00,4.00,4.00,under $5.00
999,8,4,1.00,4.00,4.00,under $5.00
1000,9,4,5.00,4.00,20.00,$20.00 and over
1001,11,4,3.00,4.00,12.00,$10.00 - $19.00


# **Aggregations**

There are the only aggregated functions available in SQL

The abreviation is **SCAMM**



*   S -> SUM
*   C -> COUNT
*   A -> AVG
*   M -> MIN
*   M -> MAX

In [9]:
 # Question -- We want to get the most and least expensive items available in the vendor’s inventory.
query = """
SELECT max(original_price) AS most_exp, min(original_price) AS min_exp
FROM `farmers_market.vendor_inventory`
"""

df = client.query(query).to_dataframe()
df

,most_exp,min_exp
0,18.0,0.5


In [10]:
# Question -- We want to calculate how much revenue has been generated in total by purchases made by our customers.
query = """
SELECT round(sum(quantity * cost_to_customer_per_qty)) AS total_revenue
FROM `farmers_market.customer_purchases`
"""

df = client.query(query).to_dataframe()
df

,total_revenue
0,8670.0


In [11]:
# Question -- We want to find out the average quantity of products purchased by the customers on the date ‘2019-05-01’.
query = """
SELECT avg(quantity) AS avg_qty
FROM `farmers_market.customer_purchases`
WHERE market_date = '2019-05-01'
"""

df = client.query(query).to_dataframe()
df

,avg_qty
0,3.5


# **COUNT Aggregation**

count(*) -> This will give the number of rows in the table even if values are Null

count(col_name) -> This will give the all the Non- Null Records in that column

count(distinct col_name) -> Thsi will give the Unique values of Non- Null records in that column

In [12]:
# Question -- : We want to know the total number of purchases that happened during the second quarter of the year 2019.
query = """
SELECT COUNT(*) as total_no_purchases
FROM `farmers_market.customer_purchases`
WHERE market_date BETWEEN '2019-04-01' AND '2019-06-30'
"""

df = client.query(query).to_dataframe()
df

,total_no_purchases
0,184


In [13]:
# Question -- What if we ask you to get the number of unique customers who made any purchases during the second quarter of the year 2019.
query = """
SELECT COUNT(distinct customer_id) as unique_customers
FROM `farmers_market.customer_purchases`
WHERE market_date BETWEEN '2019-04-01' AND '2019-06-30'
"""

df = client.query(query).to_dataframe()
df

,unique_customers
0,19


In [14]:
# Question -- Find out the details of the orders where customers purchased more than avg quantity
query = """
SELECT *
FROM `farmers_market.customer_purchases`
WHERE
  quantity > (
    SELECT round(avg(quantity), 2) AS average
    FROM `farmers_market.customer_purchases`
  )
"""

df = client.query(query).to_dataframe()
df

,product_id,vendor_id,market_date,customer_id,quantity,cost_to_customer_per_qty,transaction_time
0,1,7,2019-07-06,12,3.60,6.99,09:33:00
1,1,7,2019-07-06,14,3.04,6.99,13:05:00
2,1,7,2019-07-10,23,3.61,6.99,18:56:00
3,1,7,2019-07-13,2,4.24,6.99,09:02:00
4,1,7,2019-07-17,4,3.03,6.99,18:44:00
...,...,...,...,...,...,...,...
529,4,7,2019-06-15,1,3.00,4.00,12:56:00
530,4,7,2019-06-15,2,5.00,4.00,09:59:00
531,4,7,2019-06-15,4,5.00,4.00,12:23:00
532,4,7,2019-06-15,9,5.00,4.00,08:37:00


In [15]:
# Question -- Find out the details of the orders where customers purchased more than 1.5 times avg quantity
query = """
SELECT *
FROM `farmers_market.customer_purchases`
WHERE
  quantity > (
    SELECT (round(avg(quantity), 2)*1.5) AS average
    FROM `farmers_market.customer_purchases`
  )
"""

df = client.query(query).to_dataframe()
df

,product_id,vendor_id,market_date,customer_id,quantity,cost_to_customer_per_qty,transaction_time
0,1,7,2019-07-20,1,4.93,6.99,13:00:00
1,1,7,2019-08-07,24,4.65,6.99,18:30:00
2,1,7,2019-08-10,23,4.57,6.99,13:20:00
3,1,7,2019-08-24,16,4.48,6.99,11:00:00
4,1,7,2019-08-31,22,4.86,6.99,09:12:00
...,...,...,...,...,...,...,...
172,4,7,2019-06-05,4,5.00,4.00,17:00:00
173,4,7,2019-06-12,1,5.00,4.00,17:10:00
174,4,7,2019-06-15,2,5.00,4.00,09:59:00
175,4,7,2019-06-15,4,5.00,4.00,12:23:00


In [16]:
# Question -- Find out the details of the orders where customers purchased more than 1.5 times avg quantity of orders sold before noon
query = """
SELECT *
FROM `farmers_market.customer_purchases`
WHERE
  quantity > 1.5*(
    SELECT avg(quantity) AS average
    FROM `farmers_market.customer_purchases` where transaction_time < "12:00:00"
  )
"""

df = client.query(query).to_dataframe()
df

,product_id,vendor_id,market_date,customer_id,quantity,cost_to_customer_per_qty,transaction_time
0,1,7,2019-07-20,1,4.93,6.99,13:00:00
1,1,7,2019-08-07,24,4.65,6.99,18:30:00
2,1,7,2019-08-10,23,4.57,6.99,13:20:00
3,1,7,2019-08-31,22,4.86,6.99,09:12:00
4,1,7,2019-09-04,20,5.67,6.99,17:57:00
...,...,...,...,...,...,...,...
162,4,7,2019-06-05,4,5.00,4.00,17:00:00
163,4,7,2019-06-12,1,5.00,4.00,17:10:00
164,4,7,2019-06-15,2,5.00,4.00,09:59:00
165,4,7,2019-06-15,4,5.00,4.00,12:23:00


# **CEIL AND FLOOR IN SQL**

In [17]:
# CEIL
query = """
select ceil(10.1)
"""

df = client.query(query).to_dataframe()
df

,f0_
0,11.0


In [18]:
#FLOOR
query = """
select floor(10.99)
"""

df = client.query(query).to_dataframe()
df

,f0_
0,10.0


# **Groupby**

Principle - we have to use group by when we have non-aggregated column, aggregated column we will use group by on Non- Aggregated column

**keywords :**
*   sales by region
*   sales by each customer
*   sales per category
*   Distribution of sales for each state


*   **by,each,per,distribution etc...**





In [19]:
# Question -- Determine the age range of patients admitted to the hospital.
query = """
SELECT min(Age) AS Minimum_age, max(Age) AS Maximum_age
FROM `farmers_market.hospital`
"""

df = client.query(query).to_dataframe()
df

,Minimum_age,Maximum_age
0,18,85


In [20]:
# Question --What is the average BMI (Body Mass Index) of the patients diagnosed with Obesity?
query = """
SELECT round(avg(BMI), 2) AS Average_BMI
FROM `farmers_market.hospital`
WHERE lower(Medical_Condition) = "obesity"
"""

df = client.query(query).to_dataframe()
df

,Average_BMI
0,29.26


In [21]:
# Question --How many records do we have in our database?
query = """
SELECT COUNT(*) AS total_records FROM `farmers_market.hospital`
"""

df = client.query(query).to_dataframe()
df

,total_records
0,40235


In [22]:
# Question --What is the distribution of patients' ages across the data?
query = """
SELECT Age, COUNT(*) AS no_pateints FROM `farmers_market.hospital` GROUP BY Age ORDER BY Age
"""

df = client.query(query).to_dataframe()
df

,Age,no_pateints
0,18,583
1,19,581
2,20,567
3,21,603
4,22,599
...,...,...
63,81,596
64,82,615
65,83,579
66,84,588


In [23]:
# --Q. How do billing amounts vary based on the patient's insurance provider?
query = """
SELECT Insurance_Provider, round(sum(Billing_Amount),2) as total_billing_amount_per_Insurance_provider
FROM `farmers_market.hospital`
GROUP BY 1
ORDER BY 2 desc
"""

df = client.query(query).to_dataframe()
df

,Insurance_Provider,total_billing_amount_per_Insurance_provider
0,Blue Cross,2.082721e+08
1,Medicare,2.074576e+08
2,Cigna,2.071020e+08
3,Aetna,2.028339e+08
4,UnitedHealthcare,2.027250e+08


In [24]:
# Question -- Analyze and compare the average duration of hospitalization for various medical conditions.
query = """
SELECT Medical_Condition, round(avg(Days_Hospitalised),2) as Avg_days_in_hospital
FROM `farmers_market.hospital`
GROUP BY 1
ORDER BY 2 DESC
"""

df = client.query(query).to_dataframe()
df

,Medical_Condition,Avg_days_in_hospital
0,Asthma,15.66
1,Arthritis,15.58
2,Cancer,15.52
3,Hypertension,15.51
4,Obesity,15.49
5,Diabetes,15.39


In [25]:
# Question -- Calculate the average billing amount for cancer patients in each hospital.
query = """
SELECT Hospital, avg(Billing_Amount) AS Avg_billing_amount
FROM `farmers_market.hospital`
WHERE lower(Medical_Condition) = "cancer"
GROUP BY 1
ORDER BY 2 DESC
"""

df = client.query(query).to_dataframe()
df

,Hospital,Avg_billing_amount
0,Hernandez-Morton,52373.03
1,Ruiz-Anthony,52154.24
2,George-Gonzalez,52102.24
3,Rocha-Carter,52092.67
4,Davis PLC,51899.02
...,...,...
6228,Jones-Smith,-599.27
6229,Roberts-King,-808.47
6230,"Fitzpatrick, Nielsen and Mcdonald",-887.02
6231,Clements-Bowman,-1277.65


In [26]:
# Question -- What percentage of patients are diagnosed with each medical condition?
query = """
SELECT
  Medical_Condition,
  100 * (COUNT(*) / (SELECT COUNT(*) FROM `farmers_market.hospital`))
    AS percentage
FROM `farmers_market.hospital`
GROUP BY 1
ORDER BY 2 DESC
"""

df = client.query(query).to_dataframe()
df

,Medical_Condition,percentage
0,Arthritis,16.952902
1,Diabetes,16.890767
2,Cancer,16.622344
3,Hypertension,16.597490
4,Obesity,16.490618
5,Asthma,16.445880


In [27]:
# Question -- Total Revenue by Each product
query = """
SELECT product_id, round(sum(quantity * cost_to_customer_per_qty),2) as total_revenue
FROM `farmers_market.customer_purchases`
GROUP BY 1
ORDER BY 2 DESC
"""

df = client.query(query).to_dataframe()
df

,product_id,total_revenue
0,2,3192.52
1,1,2839.97
2,4,2086.50
3,3,551.50


In [28]:
# Above same question this time we didn't give the product_id in select clause but we group by product_id so output doesn't have the product_id column
query = """
SELECT round(sum(quantity * cost_to_customer_per_qty),2) as total_revenue
FROM `farmers_market.customer_purchases`
GROUP BY product_id
ORDER BY total_revenue DESC
"""

df = client.query(query).to_dataframe()
df

,total_revenue
0,3192.52
1,2839.97
2,2086.50
3,551.50


In [29]:
# Question -- Find the Unique products by using groupby
query = """
SELECT product_id
FROM `farmers_market.customer_purchases`
GROUP BY 1

"""

df = client.query(query).to_dataframe()
df

,product_id
0,1
1,2
2,3
3,4


In [30]:
# Question -- -- Total Revenue by each customer for each product
# C1 -- P1 -- 1000
# C1    p2 -2000
# C2 -- P1-- 500
# C2 -- P2 -- 700
query = """
SELECT
  customer_id,
  product_id,
  round(sum(quantity * cost_to_customer_per_qty),2) AS total_revenue
FROM `farmers_market.customer_purchases`
GROUP BY 1, 2
ORDER BY 1
"""

df = client.query(query).to_dataframe()
df

,customer_id,product_id,total_revenue
0,1,2,126.20
1,1,3,51.00
2,1,4,149.00
3,1,1,188.52
4,2,1,226.13
...,...,...,...
94,26,3,13.50
95,26,2,177.40
96,57,1,34.95
97,58,3,1.00


In [31]:
# Question -- Count the number of purchases each customer made per market date.

query = """
SELECT COUNT(transaction_time) AS counting, market_date, customer_id
FROM `farmers_market.customer_purchases`
GROUP BY 2, 3
ORDER BY 1 DESC
"""

df = client.query(query).to_dataframe()
df

,counting,market_date,customer_id
0,5,2019-07-13,10
1,4,2020-08-08,5
2,4,2020-09-02,7
3,4,2020-09-19,15
4,4,2020-09-26,14
...,...,...,...
672,1,2019-08-24,15
673,1,2019-06-15,4
674,1,2020-09-16,19
675,1,2019-06-15,9


In [32]:
# Question -- How many different kinds of products were purchased by each customer on each market date?

query = """
SELECT customer_id, market_date, COUNT(DISTINCT product_id) AS diff_products
FROM `farmers_market.customer_purchases`
GROUP BY 1, 2
ORDER BY 3 DESC
"""

df = client.query(query).to_dataframe()
df

,customer_id,market_date,diff_products
0,10,2019-07-13,3
1,15,2020-09-19,3
2,16,2019-07-03,3
3,7,2020-09-02,3
4,14,2020-09-26,3
...,...,...,...
672,2,2019-06-15,1
673,4,2019-06-15,1
674,8,2019-06-15,1
675,9,2019-06-15,1


In [33]:
# Question --Calculate the total price paid by customer_id 3 per market_date.

query = """
SELECT customer_id,market_date, round(sum(quantity * cost_to_customer_per_qty),2) AS total_price
FROM `farmers_market.customer_purchases`
WHERE customer_id = 3
GROUP BY 1,2
ORDER BY 3 DESC
"""

df = client.query(query).to_dataframe()
df

,market_date,total_price
0,2020-09-16,36.33
1,2019-07-31,26.62
2,2019-06-01,20.00
3,2019-04-24,20.00
4,2019-04-13,20.00
5,2019-07-10,19.34
6,2019-08-14,18.50
7,2019-09-28,17.99
8,2020-08-01,17.00
9,2019-05-01,16.00


In [34]:
# Question --What if we wanted to determine how much this customer id = 3 had spent at each vendor, regardless of date?

query = """
SELECT
  customer_id, vendor_id, round(sum(quantity * cost_to_customer_per_qty), 2) as total_price
FROM `farmers_market.customer_purchases`
WHERE customer_id = 3
GROUP BY 1, 2
ORDER BY 3 DESC
"""

df = client.query(query).to_dataframe()
df

,customer_id,vendor_id,total_price
0,3,7,340.26


In [35]:
# Question --Count how many products were for sale on each market date

query = """
SELECT market_date, COUNT(product_id) AS no_of_products
FROM `farmers_market.vendor_inventory`
GROUP BY 1
ORDER BY 2 DESC
"""

df = client.query(query).to_dataframe()
df

,market_date,no_of_products
0,2019-07-03,8
1,2019-07-06,8
2,2019-07-10,8
3,2019-07-13,8
4,2019-07-17,8
...,...,...
137,2019-05-04,4
138,2019-04-27,4
139,2020-03-04,4
140,2019-12-21,4


In [36]:
# Question --how many different products each vendor offered BETWEEN '2019-04-03' AND '2019-05-16'

query = """
SELECT vendor_id, COUNT(DISTINCT product_id) AS unique_products
FROM `farmers_market.vendor_inventory`
WHERE market_date BETWEEN '2019-04-03' AND '2019-05-16'
GROUP BY 1
ORDER BY 2 DESC
"""

df = client.query(query).to_dataframe()
df

,vendor_id,unique_products
0,8,3
1,7,1


In [37]:
# Question -- In addition to the count of different products per vendor , we also want the average original price of a product per vendor BETWEEN '2019-04-03' AND '2019-05-16'?

query = """
SELECT
  vendor_id,
  COUNT(DISTINCT product_id) AS unique_products,
  avg(original_price) AS avg_price
FROM `farmers_market.vendor_inventory`
WHERE market_date BETWEEN '2019-04-03' AND '2019-05-16'
GROUP BY 1
"""

df = client.query(query).to_dataframe()
df

,vendor_id,unique_products,avg_price
0,7,1,4.000000
1,8,3,14.166667


In [38]:
# Question -- calculate the price per item

query = """
SELECT
  vendor_id,
  COUNT(DISTINCT product_id) AS unique_products,
  avg(original_price) AS avg_price,
  round(sum(quantity*original_price)/sum(quantity),1) as avg_price_per_item
FROM `farmers_market.vendor_inventory`
WHERE market_date BETWEEN '2019-04-03' AND '2019-05-16'
GROUP BY 1
"""

df = client.query(query).to_dataframe()
df

,vendor_id,unique_products,avg_price,avg_price_per_item
0,7,1,4.000000,4.0
1,8,3,14.166667,11.2


# **Having Clause**

Principle - we have to use having on aggregated column


# **Where vs Having**

**Where Clause -**

*   Written after from clause
*   Filter all values in the table which means col must be present in table

**Having Clause -**

*   Written after group by clause
*   Having only used on aggreg column which is SCAMM it filter the aggreg col





In [39]:
#Question -- Get the list of the products with revenue for the products with total revenue > 1000
query = """
SELECT product_id, round(sum(quantity * cost_to_customer_per_qty), 1) AS total_revenue
FROM `farmers_market.customer_purchases`
GROUP BY 1
HAVING total_revenue > 1000
ORDER BY 1
"""

df = client.query(query).to_dataframe()
df

,product_id,total_revenue
0,1,2840.0
1,2,3192.5
2,4,2086.5


In [40]:
#Question -- Get the list of the products with revenue for the products with total revenue > 1000 excluding product_id = 1
query = """
SELECT product_id, round(sum(quantity * cost_to_customer_per_qty), 1) AS total_revenue
FROM `farmers_market.customer_purchases`
WHERE product_id!=1
GROUP BY 1
HAVING total_revenue > 1000
ORDER BY 1
"""

df = client.query(query).to_dataframe()
df

,product_id,total_revenue
0,2,3192.5
1,4,2086.5


In [41]:
#Question -- List the days where the qty sold is greater than overall avg qty
query = """
SELECT market_date, avg(quantity) AS avg_qty
FROM `farmers_market.customer_purchases`
GROUP BY 1
HAVING
  avg_qty > (
    SELECT avg(quantity) AS average FROM `farmers_market.customer_purchases`
  )
ORDER BY 1
"""

df = client.query(query).to_dataframe()
df

,market_date,avg_qty
0,2019-04-06,3.500000
1,2019-04-10,3.833333
2,2019-04-17,3.500000
3,2019-04-24,3.000000
4,2019-04-27,3.222222
5,2019-05-01,3.500000
6,2019-05-11,3.222222
7,2019-05-18,3.333333
8,2019-06-01,3.666667
9,2019-06-12,3.250000


In [42]:
#Question -- List the customer who have purchased all the products i.e he should have purchased all 4 products
query = """
SELECT customer_id, COUNT(DISTINCT product_id) AS no_of_prods
FROM `farmers_market.customer_purchases`
GROUP BY 1
HAVING
  no_of_prods = (
    SELECT COUNT(DISTINCT product_id) FROM `farmers_market.customer_purchases`
  )
ORDER BY 1
"""

df = client.query(query).to_dataframe()
df

,customer_id,no_of_prods
0,1,4
1,2,4
2,3,4
3,4,4
4,5,4
5,6,4
6,7,4
7,8,4
8,9,4
9,10,4


# **SUB-Query Using in from statement**

In [44]:
#Question -- Get the list of the products with revenue for the products with total revenue > 1000 without using Having
query = """
SELECT *
FROM
  (
    SELECT
      product_id,
      round(sum(quantity * cost_to_customer_per_qty), 1) AS total_revenue
    FROM `farmers_market.customer_purchases`
    GROUP BY 1
    ORDER BY 1
  ) AS temp
WHERE total_revenue > 1000
"""

df = client.query(query).to_dataframe()
df

# NOTE - Same Question we did by using having clause on cell no - 39

,product_id,total_revenue
0,1,2840.0
1,2,3192.5
2,4,2086.5
